# Baseline Interface

**Module:** Intelligent Interactive Systems

**Assignment:** Assignment 2, project report and prototype

**Name:** Leeya Hundal

**Student number:** 3183349

**Email:** LXH1038@student.bham.ac.uk

**Repository:** https://github.com/Leeya01/Assignment-A2-Project

**Version:** 2, interface with feedback, control and simulated suggestions

**Submission release:** a2-submission

**Last updated:** 16/11/26


This notebook builds the second version of the interface for a local region image editor. The user paints over part of an image, types a short instruction and only that area is changed. This version adds the before and after view, the red changed-area highlight, a seed box and guidance slider, undo, back to original, a status line and a tick list of simulated suggestions. Notebook 1 (`01-inpainting.ipynb`) holds the earlier inpainting experiments, and the errors I hit and fixed along the way are in the development log.

**To run it:** set the accelerator to GPU T4, turn Internet on, then run the cells from top to bottom: setup, model, functions, then the app cell last. It prints a `gradio.live` link.


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
!pip install -q gradio


import gradio as gr
from PIL import Image, ImageFilter
from scipy.ndimage import binary_fill_holes
import torch
from diffusers import AutoPipelineForInpainting

# use the GPU if there is one, otherwise fall back to the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("Using device:", device)

# SDXL inpainting model, loaded in half precision to fit on the T4
sdxl_id = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"
pipe_xl = AutoPipelineForInpainting.from_pretrained(
    sdxl_id, torch_dtype=dtype, variant="fp16"
)
pipe_xl = pipe_xl.to(device)
print("SDXL loaded")
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Using device: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

The config attributes {'decay': 0.9999, 'inv_gamma': 1.0, 'min_decay': 0.0, 'optimization_step': 37000, 'power': 0.6666666666666666, 'update_after_step': 0, 'use_ema_warmup': False} were passed to UNet2DConditionModel, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

SDXL loaded
/kaggle/input/datasets/leeyahundal/testimage2/Enchanted Underwater Pirate Shipwreck.png


Section 3: Editing functions.

get_mask turns the brush strokes into a mask. Strokes can sit on more than one layer in the editor, so it combines every layer. It also fills the inside of any closed loop, because testing showed that people often circle an object instead of colouring it in, which selected nothing at first.

apply_edit does the edit itself:

Checks the inputs first and gives a plain error message if the image, prompt or painted area is missing.
Shrinks the image to 512 on its longest side, keeping its shape. Running SDXL on the full-detail image made edits blend into the background, and shrinking first gave visible results. Squashing images into a square distorted landscape pictures, so the shape is kept.
Enlarges to around 1024 for SDXL, which works best at that size, then scales back down.
Pastes the result back using a softened mask, so everything outside the painted area stays identical to the original.
Measures the average change outside the mask as a precision check (0.0 means nothing outside the mask moved).

The seed and guidance are inputs so the user can ask for a different result or choose how closely the model follows the prompt.

Section 4: The app, with feedback and control.

The user uploads an image, paints over the area to change, types what should go there and presses apply. After each edit the app shows:

Before and after: the original next to the result, so the user can compare.
Changed area (red): the pixels that really changed, worked out from the difference between the two images, not from the painted mask. This was added because early edits looked as if nothing had happened, so I could not tell whether the model had failed or only blended the change in.
Mask the app used: shows what the system understood from the brush strokes.
Precision check: the change outside the mask, plus the change inside it and the share of masked pixels changed.

In [23]:
def get_mask(editor_value):
    # strokes can sit on several layers, so I combine the alpha channel of every layer
    if editor_value is None or len(editor_value["layers"]) == 0:
        return None
    painted = None
    for layer in editor_value["layers"]:
        alpha = np.array(layer)[:, :, 3] > 0
        painted = alpha if painted is None else (painted | alpha)
    # fill the inside of any closed loop, so circling an object selects it
    filled = binary_fill_holes(painted)
    return Image.fromarray(filled.astype(np.uint8) * 255)

def apply_edit(editor_value, prompt, seed, guidance):
    if editor_value is None or editor_value["background"] is None:
        raise gr.Error("Upload an image first.")
    if not prompt or not prompt.strip():
        raise gr.Error("Type what should appear in the painted area.")
    mask = get_mask(editor_value)
    if mask is None or np.array(mask).max() == 0:
        raise gr.Error("Paint over the area you want to change first.")

    # shrink to 512 on the lo
    ngest side, keeping the shape, sizes rounded to multiples of 8
    base = editor_value["background"].convert("RGB")
    scale = 512 / max(base.size)
    w = int(base.width * scale) // 8 * 8
    h = int(base.height * scale) // 8 * 8
    base = base.resize((w, h))
    mask = mask.resize((w, h), Image.NEAREST)
    soft = mask.filter(ImageFilter.GaussianBlur(5))

    # SDXL works at around 1024, so enlarge for the edit, then scale back down
    up = 1024 / max(w, h)
    big_w = int(w * up) // 8 * 8
    big_h = int(h * up) // 8 * 8

    out = pipe_xl(
        prompt=prompt,
        negative_prompt="blurry, text, watermark, realistic photo, harsh shadows",
        image=base.resize((big_w, big_h)),
        mask_image=mask.resize((big_w, big_h)),
        padding_mask_crop=64,
        guidance_scale=float(guidance),
        num_inference_steps=30,
        generator=torch.Generator(device).manual_seed(int(seed)),
    ).images[0].resize((w, h))

    # paste back so everything outside the mask matches the original
    final = Image.composite(out, base, soft)

    # precision check, average change outside the mask on a 0 to 255 scale
    outside = np.array(soft) == 0
    diff = np.abs(np.array(final).astype(float) - np.array(base).astype(float)).mean(axis=2)
    note = "Average change outside the mask: " + str(round(diff[outside].mean(), 3))
    return final, note, mask

In [24]:
try:
    demo.close()
except NameError:
    pass

def edit_with_views(editor_value, prompt, seed, guidance):
    final, note, mask = apply_edit(editor_value, prompt, seed, guidance)
    before = editor_value["background"].convert("RGB").resize(final.size)

    # difference between the two images, per pixel
    diff = np.abs(np.array(final).astype(float) - np.array(before).astype(float)).mean(axis=2)
    inside = np.array(mask) > 0

    # tint the pixels that really changed
    changed = diff > 3
    highlight = np.array(final).copy()
    red = np.array([255, 0, 0])
    highlight[changed] = (0.5 * highlight[changed] + 0.5 * red).astype(np.uint8)

    note = note + ". Average change inside the mask: " + str(round(diff[inside].mean(), 2))
    note = note + ". Share of masked pixels changed: " + str(round(100 * changed[inside].mean(), 1)) + "%"
    return before, final, Image.fromarray(highlight), note, mask

def use_result(img):
    if img is None:
        raise gr.Error("Apply an edit first.")
    return img

with gr.Blocks() as demo:
    gr.Markdown("Inpainting editor")
    editor = gr.ImageEditor(
        type="pil",
        label="Upload an image and paint over the area to change",
        brush=gr.Brush(colors=["#ffffff"], default_size=20),
    )
    prompt_box = gr.Textbox(label="What should go in the painted area?")
    seed_box = gr.Number(label="Seed (change it for a different result)", value=42, precision=0)
    guidance_box = gr.Slider(3, 12, value=5.5, step=0.5, label="How closely to follow the prompt")
    apply_btn = gr.Button("Apply edit")
    with gr.Row():
        before_view = gr.Image(label="Before")
        result = gr.Image(label="After")
    highlight_view = gr.Image(label="Changed area (red)")
    mask_view = gr.Image(label="Mask the app used")
    check = gr.Textbox(label="Precision check")
    again_btn = gr.Button("Keep editing this result")
    apply_btn.click(
        edit_with_views,
        [editor, prompt_box, seed_box, guidance_box],
        [before_view, result, highlight_view, check, mask_view],
    )
    again_btn.click(use_result, result, editor)

demo.launch(share=True, show_error=True)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://2681a9c5c41a68c24a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1963, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1963, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^

... (2 outputs left out)

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

Section 5: Undo and keep original. The app keeps a history of the images before each edit, plus the original. Undo steps back one edit and Back to original restores the first image. A status line tells the user what happened. This gives the user control over a model that doesn’t always do what was asked.

In [25]:
try:
    demo.close()
except NameError:
    pass

def edit_with_views(editor_value, prompt, seed, guidance, history, original):
    final, note, mask = apply_edit(editor_value, prompt, seed, guidance)
    start = editor_value["background"].convert("RGB")

    # remember the original when starting fresh, and the image before this edit
    if not history:
        original = start
    history = history + [start]

    before = start.resize(final.size)
    diff = np.abs(np.array(final).astype(float) - np.array(before).astype(float)).mean(axis=2)
    inside = np.array(mask) > 0

    # tint the pixels that really changed
    changed = diff > 3
    highlight = np.array(final).copy()
    red = np.array([255, 0, 0])
    highlight[changed] = (0.5 * highlight[changed] + 0.5 * red).astype(np.uint8)

    note = note + ". Average change inside the mask: " + str(round(diff[inside].mean(), 2))
    note = note + ". Share of masked pixels changed: " + str(round(100 * changed[inside].mean(), 1)) + "%"
    return before, final, Image.fromarray(highlight), note, mask, history, original

def use_result(img):
    if img is None:
        raise gr.Error("Apply an edit first.")
    return img

def undo(history):
    if not history:
        raise gr.Error("Nothing to undo.")
    previous = history[-1]
    left = len(history) - 1
    return previous, previous, history[:-1], "Undid the last edit. Edits left to undo: " + str(left)

def back_to_original(original):
    if original is None:
        raise gr.Error("Apply an edit first, so there is an original to go back to.")
    return original, original, [], "Back to the original image."

with gr.Blocks() as demo:
    gr.Markdown("Inpainting editor")
    history = gr.State([])
    original = gr.State(None)

    editor = gr.ImageEditor(
        type="pil",
        label="Upload an image and paint over the area to change",
        brush=gr.Brush(colors=["#ffffff"], default_size=20),
    )
    prompt_box = gr.Textbox(label="What should go in the painted area?")
    seed_box = gr.Number(label="Seed (change it for a different result)", value=42, precision=0)
    guidance_box = gr.Slider(3, 12, value=5.5, step=0.5, label="How closely to follow the prompt")
    apply_btn = gr.Button("Apply edit")
    with gr.Row():
        before_view = gr.Image(label="Before")
        result = gr.Image(label="After")
    highlight_view = gr.Image(label="Changed area (red)")
    mask_view = gr.Image(label="Mask the app used")
    check = gr.Textbox(label="Precision check")
    with gr.Row():
        again_btn = gr.Button("Keep editing this result")
        undo_btn = gr.Button("Undo last edit")
        original_btn = gr.Button("Back to original")
    status = gr.Textbox(label="Status")

    apply_btn.click(
        edit_with_views,
        [editor, prompt_box, seed_box, guidance_box, history, original],
        [before_view, result, highlight_view, check, mask_view, history, original],
    )
    again_btn.click(use_result, result, editor)
    undo_btn.click(undo, history, [editor, result, history, status])
    original_btn.click(back_to_original, original, [editor, result, history, status])

demo.launch(share=True, show_error=True)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://6bc05a0cf70fee4cb7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

Section 6: Simulated suggested regions, one prepared set per demo image. Each set is a small hand-made list of labels, boxes and prompts, so the suggestions only make sense for the image they were made for. Nothing here is detected by a model. Planned: use a segmentation or detection model to find objects in any uploaded image. Boxes are fractions of the image width and height.

In [36]:
import json
from PIL import ImageDraw

# simulated suggestions: prepared by hand for each demo image, not detected by a model
suggestion_sets = {
    "Underwater scene": [
        {
            "label": "Treasure chest looks plain",
            "box": [0.64, 0.63, 0.85, 0.85],
            "prompt": "an open treasure chest overflowing with gold coins, detailed fantasy illustration, same lighting as the scene",
        },
        {
            "label": "Large crystal could be a different colour",
            "box": [0.86, 0.22, 1.0, 0.62],
            "prompt": "a cluster of glowing pink crystals, detailed fantasy illustration, same underwater lighting",
        },
        {
            "label": "Open water could have wildlife",
            "box": [0.45, 0.03, 0.78, 0.30],
            "prompt": "a sea turtle swimming, detailed fantasy illustration, soft light rays from above",
        },
    ],
    "Cosy study": [
        {
            "label": "Mug could be a different colour",
            "box": [0.20, 0.72, 0.42, 0.92],
            "prompt": "a plain blue ceramic mug on a wooden desk, soft flat cartoon illustration, warm sunset lighting",
        },
        {
            "label": "Plant could be a different plant",
            "box": [0.70, 0.38, 0.98, 0.73],
            "prompt": "a tall cactus in a white pot, soft pastel illustration, warm sunset lighting",
        },
        {
            "label": "Wall poster could show something else",
            "box": [0.47, 0.03, 0.72, 0.50],
            "prompt": "a framed poster of a mountain at sunrise, soft pastel illustration",
        },
    ],
}

with open("suggestions.json", "w") as f:
    json.dump(suggestion_sets, f, indent=2)

def labels_for(set_name):
    # numbered labels so they match the numbers drawn on the boxes
    return [str(i + 1) + ". " + s["label"] for i, s in enumerate(suggestion_sets[set_name])]

print("Saved", len(suggestion_sets), "sets")

Saved 2 sets


Section 7: Tick list of simulated suggestions. The user picks a demo image set, and the tick list shows that set’s suggestions, all ticked. They can untick any and apply the rest, each applied in turn through the same edit step as the brush, so the paste-back guarantee still holds. A button draws the boxes on the image first, so the user can see what each suggestion refers to and check the set matches the uploaded image. The whole batch is one history step, so Undo reverses all of it. The suggestions are simulated and labelled as such in the app.

In [37]:
try:
    demo.close()
except NameError:
    pass

def edit_with_views(editor_value, prompt, seed, guidance, history, original):
    final, note, mask = apply_edit(editor_value, prompt, seed, guidance)
    start = editor_value["background"].convert("RGB")

    # remember the original when starting fresh, and the image before this edit
    if not history:
        original = start
    history = history + [start]

    before = start.resize(final.size)
    diff = np.abs(np.array(final).astype(float) - np.array(before).astype(float)).mean(axis=2)
    inside = np.array(mask) > 0

    # tint the pixels that really changed
    changed = diff > 3
    highlight = np.array(final).copy()
    red = np.array([255, 0, 0])
    highlight[changed] = (0.5 * highlight[changed] + 0.5 * red).astype(np.uint8)

    note = note + ". Average change inside the mask: " + str(round(diff[inside].mean(), 2))
    note = note + ". Share of masked pixels changed: " + str(round(100 * changed[inside].mean(), 1)) + "%"
    return before, final, Image.fromarray(highlight), note, mask, history, original

def show_suggestions(editor_value, set_name):
    if editor_value is None or editor_value["background"] is None:
        raise gr.Error("Upload an image first.")
    img = editor_value["background"].convert("RGB")
    draw = ImageDraw.Draw(img)
    for i, s in enumerate(suggestion_sets[set_name]):
        x0, y0, x1, y1 = s["box"]
        box = [x0 * img.width, y0 * img.height, x1 * img.width, y1 * img.height]
        draw.rectangle(box, outline="red", width=4)
        draw.text((box[0] + 8, box[1] + 8), str(i + 1), fill="red")
    return img

def apply_suggestions(editor_value, chosen, set_name, seed, guidance, history, original):
    if editor_value is None or editor_value["background"] is None:
        raise gr.Error("Upload an image first.")
    if not chosen:
        raise gr.Error("Tick at least one suggestion.")

    start = editor_value["background"].convert("RGB")
    if not history:
        original = start
    history = history + [start]

    current = start
    for label in chosen:
        # the number at the start of the label says which suggestion it is
        s = suggestion_sets[set_name][int(label.split(".")[0]) - 1]
        x0, y0, x1, y1 = s["box"]
        # turn the box into a painted layer, so the normal edit step can use it
        layer = np.zeros((current.height, current.width, 4), dtype=np.uint8)
        layer[int(y0 * current.height):int(y1 * current.height),
              int(x0 * current.width):int(x1 * current.width), 3] = 255
        fake_editor = {"background": current, "layers": [Image.fromarray(layer)]}
        current, note, mask = apply_edit(fake_editor, s["prompt"], seed, guidance)

    before = start.resize(current.size)
    diff = np.abs(np.array(current).astype(float) - np.array(before).astype(float)).mean(axis=2)
    changed = diff > 3
    highlight = np.array(current).copy()
    red = np.array([255, 0, 0])
    highlight[changed] = (0.5 * highlight[changed] + 0.5 * red).astype(np.uint8)

    skipped = len(labels_for(set_name)) - len(chosen)
    status_text = "Applied " + str(len(chosen)) + " suggestion(s), skipped " + str(skipped) + ". These suggestions are simulated."
    return before, current, Image.fromarray(highlight), note, mask, history, original, status_text

def change_set(set_name):
    labels = labels_for(set_name)
    return gr.CheckboxGroup(choices=labels, value=labels)

def use_result(img):
    if img is None:
        raise gr.Error("Apply an edit first.")
    return img

def undo(history):
    if not history:
        raise gr.Error("Nothing to undo.")
    previous = history[-1]
    left = len(history) - 1
    return previous, previous, history[:-1], "Undid the last edit. Edits left to undo: " + str(left)

def back_to_original(original):
    if original is None:
        raise gr.Error("Apply an edit first, so there is an original to go back to.")
    return original, original, [], "Back to the original image."

first_set = list(suggestion_sets.keys())[0]

with gr.Blocks() as demo:
    gr.Markdown("Inpainting editor")
    history = gr.State([])
    original = gr.State(None)

    editor = gr.ImageEditor(
        type="pil",
        label="Upload an image and paint over the area to change",
        brush=gr.Brush(colors=["#ffffff"], default_size=20),
    )
    prompt_box = gr.Textbox(label="What should go in the painted area?")
    seed_box = gr.Number(label="Seed (change it for a different result)", value=42, precision=0)
    guidance_box = gr.Slider(3, 12, value=5.5, step=0.5, label="How closely to follow the prompt")
    apply_btn = gr.Button("Apply edit")

    gr.Markdown("Suggested edits (simulated: prepared by hand for the demo images, not detected by a model)")
    set_dropdown = gr.Dropdown(choices=list(suggestion_sets.keys()), value=first_set, label="Which demo image are the suggestions for?")
    tick_list = gr.CheckboxGroup(choices=labels_for(first_set), value=labels_for(first_set), label="Untick any you do not want")
    with gr.Row():
        show_btn = gr.Button("Show suggested regions")
        suggest_btn = gr.Button("Apply ticked suggestions")
    suggestion_view = gr.Image(label="Suggested regions")

    with gr.Row():
        before_view = gr.Image(label="Before")
        result = gr.Image(label="After")
    highlight_view = gr.Image(label="Changed area (red)")
    mask_view = gr.Image(label="Mask the app used")
    check = gr.Textbox(label="Precision check")
    with gr.Row():
        again_btn = gr.Button("Keep editing this result")
        undo_btn = gr.Button("Undo last edit")
        original_btn = gr.Button("Back to original")
    status = gr.Textbox(label="Status")

    apply_btn.click(
        edit_with_views,
        [editor, prompt_box, seed_box, guidance_box, history, original],
        [before_view, result, highlight_view, check, mask_view, history, original],
    )
    set_dropdown.change(change_set, set_dropdown, tick_list)
    show_btn.click(show_suggestions, [editor, set_dropdown], suggestion_view)
    suggest_btn.click(
        apply_suggestions,
        [editor, tick_list, set_dropdown, seed_box, guidance_box, history, original],
        [before_view, result, highlight_view, check, mask_view, history, original, status],
    )
    again_btn.click(use_result, result, editor)
    undo_btn.click(undo, history, [editor, result, history, status])
    original_btn.click(back_to_original, original, [editor, result, history, status])

demo.launch(share=True, show_error=True)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8c69e1e9490cd6842c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Development log: errors, causes and fixes

These are the problems I hit while building the interface, what caused each one and what I changed. Most were found by testing the app myself and checking the feedback boxes I added.

Setup and running

NameError: name 'gr' is not defined: the Gradio import was missing from the cell, or the kernel had restarted. Added the install and import lines to the top of the cell.
NameError: name 'pipe_xl' is not defined: the interface is in a new notebook, so the SDXL model from Day 1 was not loaded. Added a cell that loads it.
NameError: name 'apply_edit' is not defined: the kernel restarted and everything defined earlier was lost. Reran the cells in order: setup, model, functions, app.
127.0.0.1 refused to connect: the local link points at the Kaggle server, not my computer. Only the gradio.live share link works from Kaggle.
Cell never finished, and I had to stop it before running another: debug=True makes the launch cell wait for the app. Removed it and used show_error=True instead.
TypeError: apply_edit() takes 2 positional arguments but 4 were given: an old baseline cell had been rerun and replaced the newer function. Deleted the baseline cell so only one version exists.
IncompleteInputError on launch: a closing bracket was lost when pasting. Fixed the last line.
Warnings I checked and ignored: the pip gym version conflict (nothing in this project uses it), the torch_dtype deprecation notice, the Hugging Face token warning (no token is needed) and the SDXL config attributes message (unused training settings).

Interface and mask

Circling an object changed nothing: a ring drawn around the mug left the mug unpainted, so the mask only covered the ring. The mask now fills any closed loop.
Mask stayed in the wrong place after repainting: I suspected strokes on several layers and changed the mask to combine every layer. That turned out not to be the cause (I had kept the same patch on purpose), but it is harmless, so I kept it.
“Paint over the area you want to change first”: my own message, shown when no strokes are found. It appears after “Keep editing this result”, because the canvas comes back clean.

Edit quality

Landscape images came out squashed: everything was forced into a 512 square. The original shape is now kept.
Edits looked as if nothing had happened: I could not tell whether the model had failed or had blended the change in. Added the red changed-area view and the inside and outside numbers. They showed real changes (about 22 average inside the mask, 0.0 outside), just not the ones I asked for.
Highlight missed subtle changes: the first threshold for counting a pixel as changed was too high, so blended edits showed little red. Lowered it so smaller changes show up.
Full-detail 1024 images made edits blend away: the first working version shrank to 512 and then enlarged to 1024. Running SDXL on the full-detail image gave much weaker edits. I restored that approach with the original shape kept, and also reset guidance to 5.5 and the negative prompt to the earlier version, so I cannot say which of these changes mattered most.
Adding a new object to empty sand was unreliable: barrels, crystals and similar prompts were blended into the background or became a fish or coral. Replacing an object that was already there worked more reliably, and a sea turtle in open water worked well. Kept the failed attempts as evidence.

Control and suggestions

Undo gave no sign that anything had happened: added a history of earlier images, an undo button, a back to original button and a status line that says what happened and how many edits are left to undo.
Hard-coded suggestions only made sense for one image: a treasure chest suggestion means nothing on a photo of a basketball court. I made one prepared set per demo image, with a dropdown to choose, and a button that draws the suggested boxes so the user can check them against the picture. They stay simulated and are labelled that way.
Real detection is planned, not built: an object detector would find regions in any image. I decided to keep the suggestions simulated and list the detector as planned, so the report does not claim detection that does not exist.

Summary

I built a Gradio interface for editing one region of an image with SDXL inpainting. The user paints an area, describes what should go there and gets back an image in which nothing outside the painted area has changed (average change 0.0). Most problems came from setup (lost kernel state, old cells overwriting new ones) or from the model not doing what I asked. I handled the second kind by adding feedback instead of just retrying: a red view of what really changed, numbers inside and outside the mask, undo and a way back to the original. Testing showed the model reworks existing objects and open water reliably but often fails to add new objects to busy or empty ground, so I kept those failures as evidence. The suggested edits are simulated, prepared by hand for two demo images, and real object detection is planned.

If you want a shorter version for the README’s development record, say so and I’ll cut it to a handful of bullets.